# MedGemma 4B QLoRA Fine-Tuning Pipeline

This notebook implements deliverables:

1. Base-model evaluation  
2. QLoRA smoke test  
3. Full-dataset QLoRA profile for an available CUDA GPU  
4. Post-fine-tuning evaluation  
5. Clinical safety metrics  
6. Hardware report

The data is synthetic and MIMIC-IV-ED-shaped. Results demonstrate technical feasibility only and are not clinical validation.


## Execution Controls

The notebook auto-detects whether CUDA is available:

- Base and adapter evaluation will use GPU if available, otherwise CPU. CPU evaluation can be very slow.
- QLoRA training requires a CUDA GPU. It is not locked to a specific GPU model.
- If a larger profile runs out of memory, use a smaller profile or reduce `max_length`, `lora_r`, or row limits.

Small test:

```python
PROFILE = "smoke"
```

Larger test::

```python
PROFILE = "larger_smoke"
```

If that succeeds and you want a stronger non-full run:

```python
PROFILE = "larger_smoke_more_steps"
```

For the full prepared training split:

```python
PROFILE = "full"
```


In [74]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import json
import os
import platform
import re
import sys
import time

import pandas as pd
import torch
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "finetune_smoke" / "train.jsonl").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from the oss_model_clinical_triage_demo repository.")


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
PROJECT_ROOT

WindowsPath('c:/Users/money/OneDrive/Documents/GitHub/oss_model_clinical_triage_demo')

In [75]:
MODEL_ID = "google/medgemma-1.5-4b-it"
LOCAL_FILES_ONLY = True

PROFILE = "larger_smoke_more_steps"  # smoke, larger_smoke, larger_smoke_more_steps, or full
DEVICE_KIND = "cuda" if torch.cuda.is_available() else "cpu"
REVIEW_CONFIDENCE_THRESHOLD = 0.60
FORCE_GPU_ONLY_LOAD = True

RUN_BASE_EVALUATION = True
RUN_TRAINING = True
RUN_ADAPTER_EVALUATION = True

PROFILES = {
    "smoke": {
        "data_dir": "data/finetune_smoke",
        "output_dir": "outputs/notebook_smoke",
        "train_limit": 20,
        "validation_limit": 10,
        "test_limit": 10,
        "max_steps": 1,
        "epochs": 1,
        "batch_size": 1,
        "gradient_accumulation": 1,
        "max_length": 256,
        "learning_rate": 2e-4,
        "lora_r": 4,
        "lora_alpha": 8,
        "eval_strategy": "no",
        "save_strategy": "steps",
        "save_steps": 1,
    },
    "larger_smoke": {
        "data_dir": "data/finetune_smoke",
        "output_dir": "outputs/notebook_larger_smoke",
        "train_limit": 500,
        "validation_limit": 100,
        "test_limit": 100,
        "max_steps": 20,
        "epochs": 1,
        "batch_size": 1,
        "gradient_accumulation": 8,
        "max_length": 512,
        "learning_rate": 2e-4,
        "lora_r": 8,
        "lora_alpha": 16,
        "eval_strategy": "steps",
        "save_strategy": "steps",
        "save_steps": 10,
    },
    "larger_smoke_more_steps": {
        "data_dir": "data/finetune_smoke",
        "output_dir": "outputs/notebook_larger_smoke_more_steps",
        "train_limit": 500,
        "validation_limit": 100,
        "test_limit": 100,
        "max_steps": 60,
        "epochs": 1,
        "batch_size": 1,
        "gradient_accumulation": 8,
        "max_length": 512,
        "learning_rate": 2e-4,
        "lora_r": 8,
        "lora_alpha": 16,
        "eval_strategy": "steps",
        "save_strategy": "steps",
        "save_steps": 20,
    },
    "full": {
        "data_dir": "data/finetune",
        "output_dir": "outputs/notebook_full",
        "train_limit": None,
        "validation_limit": 500,
        "test_limit": 1000,
        "max_steps": -1,
        "epochs": 1,
        "batch_size": 1,
        "gradient_accumulation": 8,
        "max_length": 512,
        "learning_rate": 2e-4,
        "lora_r": 8,
        "lora_alpha": 16,
        "eval_strategy": "epoch",
        "save_strategy": "epoch",
        "save_steps": 500,
    },
}

if PROFILE not in PROFILES:
    raise ValueError(f"Unknown profile: {PROFILE}")

CONFIG = dict(PROFILES[PROFILE])
CONFIG["device_kind"] = DEVICE_KIND
CONFIG["review_confidence_threshold"] = REVIEW_CONFIDENCE_THRESHOLD
CONFIG["force_gpu_only_load"] = FORCE_GPU_ONLY_LOAD
DATA_DIR = PROJECT_ROOT / CONFIG["data_dir"]
OUTPUT_DIR = PROJECT_ROOT / CONFIG["output_dir"]
ADAPTER_DIR = OUTPUT_DIR / "adapter"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.Series(CONFIG, name=PROFILE)


data_dir                                            data/finetune_smoke
output_dir                     outputs/notebook_larger_smoke_more_steps
train_limit                                                         500
validation_limit                                                    100
test_limit                                                          100
max_steps                                                            60
epochs                                                                1
batch_size                                                            1
gradient_accumulation                                                 8
max_length                                                          512
learning_rate                                                    0.0002
lora_r                                                                8
lora_alpha                                                           16
eval_strategy                                                   

## Load Prepared Data

The train, validation, and test files were split by `subject_id` in steps 1-4. The notebook verifies that the selected records remain subject-separated.

In [76]:
def load_jsonl(path: Path) -> list[dict]:
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def apply_limit(records: list[dict], limit: int | None) -> list[dict]:
    return records if limit is None else records[:limit]


train_records = apply_limit(load_jsonl(DATA_DIR / "train.jsonl"), CONFIG["train_limit"])
validation_records = apply_limit(
    load_jsonl(DATA_DIR / "validation.jsonl"),
    CONFIG["validation_limit"],
)
test_records = apply_limit(load_jsonl(DATA_DIR / "test.jsonl"), CONFIG["test_limit"])

subject_sets = {
    "train": {record["subject_id"] for record in train_records},
    "validation": {record["subject_id"] for record in validation_records},
    "test": {record["subject_id"] for record in test_records},
}
assert subject_sets["train"].isdisjoint(subject_sets["validation"])
assert subject_sets["train"].isdisjoint(subject_sets["test"])
assert subject_sets["validation"].isdisjoint(subject_sets["test"])

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train_records), len(validation_records), len(test_records)],
        "subjects": [len(subject_sets[name]) for name in ("train", "validation", "test")],
    }
)
display(split_summary)
print("Verified: selected splits have no subject overlap.")

,split,rows,subjects
0,train,500,135
1,validation,100,16
2,test,100,20


Verified: selected splits have no subject overlap.


## Shared Evaluation and Safety Metrics

The same generation and scoring functions are used for the base model and the adapter.

For ESI, a larger number means lower urgency:

- `prediction > actual`: under-triage
- `prediction < actual`: over-triage
- true ESI 1-2 predicted as ESI 4-5: severe under-triage

In [77]:
from sklearn.metrics import confusion_matrix, f1_score, recall_score
from tqdm.auto import tqdm


def parse_esi_prediction(text: str):
    match = re.search(r"(uncertain|[1-5])", str(text), flags=re.IGNORECASE)
    if not match:
        return None
    value = match.group(1).lower()
    return "uncertain" if value == "uncertain" else int(value)


def needs_human_review(prediction, confidence: float | None) -> tuple[bool, str]:
    if prediction is None:
        return True, "invalid_output"
    if prediction == "uncertain":
        return True, "model_uncertain"
    if confidence is not None and confidence < REVIEW_CONFIDENCE_THRESHOLD:
        return True, "low_confidence"
    return False, ""


def test_fingerprint(records: list[dict]) -> str:
    keys = [
        f"{record['subject_id']}:{record['stay_id']}:{record['label']}"
        for record in records
    ]
    return hashlib.sha256("|".join(keys).encode("utf-8")).hexdigest()


def compute_clinical_metrics(predictions: list[dict]) -> dict:
    total = len(predictions)
    actual = [row["actual"] for row in predictions]
    predicted = [row["prediction"] for row in predictions]
    numeric_prediction = [value if isinstance(value, int) else 0 for value in predicted]
    valid_mask = [isinstance(value, int) and 1 <= value <= 5 for value in predicted]

    correct = sum(
        prediction == label
        for prediction, label in zip(predicted, actual)
    )
    under_triage = sum(
        isinstance(prediction, int) and prediction > label
        for prediction, label in zip(predicted, actual)
    )
    over_triage = sum(
        isinstance(prediction, int) and prediction < label
        for prediction, label in zip(predicted, actual)
    )
    severe_under_triage = sum(
        label in (1, 2) and isinstance(prediction, int) and prediction in (4, 5)
        for prediction, label in zip(predicted, actual)
    )
    uncertain = sum(value == "uncertain" for value in predicted)
    invalid = sum(value is None for value in predicted)
    human_review = sum(
        bool(row.get("needs_human_review", row["prediction"] in {None, "uncertain"}))
        for row in predictions
    )
    low_confidence_review = sum(
        row.get("review_reason") == "low_confidence"
        for row in predictions
    )
    confidence_values = [
        row.get("prediction_confidence")
        for row in predictions
        if row.get("prediction_confidence") is not None
    ]

    recalls = recall_score(
        actual,
        numeric_prediction,
        labels=[1, 2, 3, 4, 5],
        average=None,
        zero_division=0,
    )
    matrix = confusion_matrix(actual, numeric_prediction, labels=[1, 2, 3, 4, 5])

    return {
        "examples": total,
        "valid_predictions": int(sum(valid_mask)),
        "coverage": sum(valid_mask) / total if total else 0.0,
        "accuracy": correct / total if total else 0.0,
        "macro_f1": f1_score(
            actual,
            numeric_prediction,
            labels=[1, 2, 3, 4, 5],
            average="macro",
            zero_division=0,
        ),
        "under_triage_rate": under_triage / total if total else 0.0,
        "over_triage_rate": over_triage / total if total else 0.0,
        "severe_under_triage_rate": severe_under_triage / total if total else 0.0,
        "uncertainty_rate": uncertain / total if total else 0.0,
        "invalid_output_rate": invalid / total if total else 0.0,
        "human_review_rate": human_review / total if total else 0.0,
        "low_confidence_review_rate": low_confidence_review / total if total else 0.0,
        "average_prediction_confidence": (
            sum(confidence_values) / len(confidence_values)
            if confidence_values
            else None
        ),
        "review_confidence_threshold": REVIEW_CONFIDENCE_THRESHOLD,
        "recall_by_esi": {
            str(label): float(value)
            for label, value in zip([1, 2, 3, 4, 5], recalls)
        },
        "confusion_matrix_labels_1_to_5": matrix.tolist(),
    }


def save_evaluation(name: str, predictions: list[dict]) -> dict:
    result = {
        "model_id": MODEL_ID,
        "profile": PROFILE,
        "test_fingerprint": test_fingerprint(test_records),
        "metrics": compute_clinical_metrics(predictions),
        "predictions": predictions,
    }
    json_path = OUTPUT_DIR / f"{name}.json"
    csv_path = OUTPUT_DIR / f"{name}.csv"
    json_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    pd.DataFrame(predictions).to_csv(csv_path, index=False)
    print(f"Saved {json_path}")
    return result


def display_metrics(result: dict):
    metrics = result["metrics"]
    scalar_metrics = {
        key: value
        for key, value in metrics.items()
        if key not in {"recall_by_esi", "confusion_matrix_labels_1_to_5"}
    }
    display(pd.DataFrame([scalar_metrics]))
    display(
        pd.DataFrame(
            metrics["confusion_matrix_labels_1_to_5"],
            index=[f"actual_{i}" for i in range(1, 6)],
            columns=[f"predicted_{i}" for i in range(1, 6)],
        )
    )
    display(
        pd.DataFrame.from_dict(
            metrics["recall_by_esi"],
            orient="index",
            columns=["recall"],
        )
    )


In [78]:
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    BitsAndBytesConfig,
)


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def release_model_memory():
    for name in (
        "base_model",
        "base_processor",
        "adapter_model",
        "adapter_processor",
        "trainer",
        "model",
        "processor",
    ):
        globals().pop(name, None)
    cleanup_cuda()


def print_gpu_memory(label: str):
    if not torch.cuda.is_available():
        print(f"{label}: CUDA not available")
        return
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"{label}: {allocated:.2f} GiB allocated, {reserved:.2f} GiB reserved, {total:.2f} GiB total")


def first_model_device(model) -> torch.device:
    for device in getattr(model, "hf_device_map", {}).values():
        if device not in {"cpu", "disk"}:
            return torch.device(device)
    return next(model.parameters()).device


def cuda_compute_dtype():
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def quantization_config():
    compute_dtype = cuda_compute_dtype()
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_quant_storage=compute_dtype,
    )


def model_load_kwargs(for_training: bool = False) -> dict:
    if torch.cuda.is_available():
        kwargs = {
            "quantization_config": quantization_config(),
            "dtype": cuda_compute_dtype(),
            "attn_implementation": "eager",
        }
        kwargs["device_map"] = {"": 0} if FORCE_GPU_ONLY_LOAD else "auto"
        return kwargs

    if for_training:
        raise RuntimeError(
            "QLoRA training requires a CUDA GPU. No CUDA GPU was detected."
        )

    return {
        "dtype": torch.float32,
        "low_cpu_mem_usage": True,
        "attn_implementation": "eager",
    }


def load_inference_model(adapter_dir: Path | None = None):
    release_model_memory()
    print_gpu_memory("Before model load")

    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        local_files_only=LOCAL_FILES_ONLY,
    )
    processor.tokenizer.padding_side = "right"

    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        local_files_only=LOCAL_FILES_ONLY,
        **model_load_kwargs(for_training=False),
    )
    if adapter_dir is not None:
        from peft import PeftModel

        model = PeftModel.from_pretrained(model, str(adapter_dir))
    model.eval()
    print_gpu_memory("After model load")
    return model, processor


def blocked_generation_tokens(tokenizer) -> list[list[int]]:
    blocked = []
    for token in ("<unused94>", "thought"):
        token_ids = tokenizer.encode(token, add_special_tokens=False)
        if token_ids:
            blocked.append(token_ids)
    return blocked


def generate_esi(model, tokenizer, user_prompt: str):
    user_prompt = f"{user_prompt}\n\nAnswer with only the value for predicted_esi_level."
    chat_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_prompt}],
        add_generation_prompt=True,
        tokenize=False,
    ) + "predicted_esi_level:"

    inputs = tokenizer(chat_text, return_tensors="pt", return_dict=True)
    device = first_model_device(model)
    inputs = {
        key: value.to(device) if hasattr(value, "to") else value
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            bad_words_ids=blocked_generation_tokens(tokenizer),
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=[
                tokenizer.eos_token_id,
                tokenizer.convert_tokens_to_ids("<end_of_turn>"),
            ],
            return_dict_in_generate=True,
            output_scores=True,
        )

    new_token_start = inputs["input_ids"].shape[-1]
    generated = tokenizer.decode(
        output.sequences[0][new_token_start:],
        skip_special_tokens=True,
    ).strip()

    confidence = None
    if output.scores:
        first_token_id = int(output.sequences[0][new_token_start].item())
        first_token_scores = output.scores[0][0].float()
        confidence = float(torch.softmax(first_token_scores, dim=-1)[first_token_id].item())

    return parse_esi_prediction(generated), generated, confidence


def evaluate_records(model, tokenizer, records: list[dict], description: str):
    predictions = []
    for record in tqdm(records, desc=description, unit="case"):
        user_prompt = record["messages"][0]["content"]
        prediction, raw_response, confidence = generate_esi(model, tokenizer, user_prompt)
        review, review_reason = needs_human_review(prediction, confidence)
        predictions.append(
            {
                "subject_id": record["subject_id"],
                "stay_id": record["stay_id"],
                "actual": record["label"],
                "prediction": prediction,
                "prediction_confidence": confidence,
                "needs_human_review": review,
                "review_reason": review_reason,
                "raw_response": raw_response,
            }
        )
    return predictions


## 5. Base-Model Evaluation

In [79]:
BASE_EVALUATION_PATH = OUTPUT_DIR / "base_evaluation.json"

if RUN_BASE_EVALUATION:
    release_model_memory()
    base_model, base_processor = load_inference_model()
    base_predictions = evaluate_records(
        base_model,
        base_processor.tokenizer,
        test_records,
        "Base MedGemma",
    )
    base_result = save_evaluation("base_evaluation", base_predictions)
    display_metrics(base_result)
    release_model_memory()
elif BASE_EVALUATION_PATH.exists():
    base_result = json.loads(BASE_EVALUATION_PATH.read_text(encoding="utf-8"))
    print("Loaded existing profile-specific base evaluation.")
    display_metrics(base_result)
else:
    print("Set RUN_BASE_EVALUATION = True and rerun this cell.")


Before model load: 3.79 GiB allocated, 5.46 GiB reserved, 7.96 GiB total


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

c:\Users\money\anaconda3\envs\meddemo\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


After model load: 6.81 GiB allocated, 6.83 GiB reserved, 7.96 GiB total


Base MedGemma:   0%|          | 0/100 [00:00<?, ?case/s]

c:\Users\money\anaconda3\envs\meddemo\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Saved c:\Users\money\OneDrive\Documents\GitHub\oss_model_clinical_triage_demo\outputs\notebook_larger_smoke_more_steps\base_evaluation.json


,examples,valid_predictions,coverage,accuracy,macro_f1,under_triage_rate,over_triage_rate,severe_under_triage_rate,uncertainty_rate,invalid_output_rate,human_review_rate,low_confidence_review_rate,average_prediction_confidence,review_confidence_threshold
0,100,100,1.0,0.2,0.084211,0.4,0.4,0.25,0.0,0.0,0.0,0.0,0.90454,0.6


,predicted_1,predicted_2,predicted_3,predicted_4,predicted_5
actual_1,0,0,0,0,20
actual_2,0,0,15,0,5
actual_3,0,0,20,0,0
actual_4,0,0,20,0,0
actual_5,0,0,20,0,0


,recall
1,0.0
2,0.0
3,1.0
4,0.0
5,0.0


## 6-7. QLoRA Smoke Test and Full-Dataset Profile

This remains QLoRA in every profile; `full` means the full prepared training split, not full-parameter fine-tuning.

Training is GPU-model agnostic. The notebook uses whatever CUDA GPU PyTorch detects. If the selected profile runs out of VRAM, switch to `smoke` or lower the row limits, sequence length, or LoRA rank.

The LoRA regular expression was validated against MedGemma's module names. It targets only Gemma language layers and excludes the vision tower.


In [80]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer


def to_sft_dataset(records: list[dict]) -> Dataset:
    return Dataset.from_list(
        [
            {
                "prompt": [record["messages"][0]],
                "completion": [record["messages"][1]],
            }
            for record in records
        ]
    )


train_dataset = to_sft_dataset(train_records)
validation_dataset = to_sft_dataset(validation_records)

display(train_dataset)
print(train_dataset[0])

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 500
})

{'prompt': [{'role': 'user', 'content': 'Estimate the Emergency Severity Index (ESI) level from this triage record. Return only JSON with the key predicted_esi_level.\n\nPatient data:\n{"gender": "F", "race": "UNKNOWN", "arrival_transport": "UNKNOWN", "temperature": 36.5, "heartrate": 79.0, "resprate": 17.0, "o2sat": 97.5, "sbp": 132.0, "dbp": 73.0, "pain": 2.0, "chiefcomplaint": "Cough"}'}], 'completion': [{'role': 'assistant', 'content': '{"predicted_esi_level": 4}'}]}


In [81]:
LANGUAGE_LORA_TARGETS = (
    r".*language_model\.layers\.\d+\."
    r"(?:self_attn\.(?:q_proj|k_proj|v_proj|o_proj)|"
    r"mlp\.(?:gate_proj|up_proj|down_proj))$"
)


def hardware_snapshot() -> dict:
    snapshot = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "platform": platform.platform(),
        "python": sys.version,
        "pytorch": torch.__version__,
        "cuda_build": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
    }
    try:
        import transformers
        import trl
        import peft
        import bitsandbytes

        snapshot["packages"] = {
            "transformers": transformers.__version__,
            "trl": trl.__version__,
            "peft": peft.__version__,
            "bitsandbytes": bitsandbytes.__version__,
        }
    except ImportError:
        pass

    try:
        import psutil

        snapshot["system_ram_gib"] = psutil.virtual_memory().total / 1024**3
    except ImportError:
        snapshot["system_ram_gib"] = None

    if torch.cuda.is_available():
        properties = torch.cuda.get_device_properties(0)
        snapshot["gpu"] = {
            "name": properties.name,
            "total_vram_gib": properties.total_memory / 1024**3,
            "compute_capability": list(torch.cuda.get_device_capability(0)),
            "allocated_gib": torch.cuda.memory_allocated(0) / 1024**3,
            "reserved_gib": torch.cuda.memory_reserved(0) / 1024**3,
            "peak_allocated_gib": torch.cuda.max_memory_allocated(0) / 1024**3,
            "peak_reserved_gib": torch.cuda.max_memory_reserved(0) / 1024**3,
        }
    return snapshot


if RUN_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "QLoRA training requires a CUDA GPU. "
            "Run base evaluation on CPU, or use a machine with CUDA for training."
        )

    gpu_properties = torch.cuda.get_device_properties(0)
    total_vram_gb = gpu_properties.total_memory / 1024**3
    print(f"Training on {gpu_properties.name} with {total_vram_gb:.1f} GiB VRAM.")

    release_model_memory()
    torch.cuda.reset_peak_memory_stats()
    training_started = time.perf_counter()

    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        local_files_only=LOCAL_FILES_ONLY,
    )
    processor.tokenizer.padding_side = "right"

    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        local_files_only=LOCAL_FILES_ONLY,
        **model_load_kwargs(for_training=True),
    )
    model.config.use_cache = False

    peft_config = LoraConfig(
        r=CONFIG["lora_r"],
        lora_alpha=CONFIG["lora_alpha"],
        lora_dropout=0.05,
        bias="none",
        target_modules=LANGUAGE_LORA_TARGETS,
        task_type="CAUSAL_LM",
    )

    training_args = SFTConfig(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=CONFIG["epochs"],
        max_steps=CONFIG["max_steps"],
        per_device_train_batch_size=CONFIG["batch_size"],
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=CONFIG["gradient_accumulation"],
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        learning_rate=CONFIG["learning_rate"],
        warmup_ratio=0.03,
        max_grad_norm=0.3,
        optim="paged_adamw_8bit",
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        max_length=CONFIG["max_length"],
        completion_only_loss=True,
        eval_strategy=CONFIG["eval_strategy"],
        save_strategy=CONFIG["save_strategy"],
        save_steps=CONFIG["save_steps"],
        logging_steps=1,
        logging_first_step=True,
        report_to="none",
        seed=42,
        dataset_num_proc=1,
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=(
            validation_dataset
            if CONFIG["eval_strategy"] != "no"
            else None
        ),
        peft_config=peft_config,
        processing_class=processor.tokenizer,
    )

    trainable_parameters = sum(
        parameter.numel()
        for parameter in trainer.model.parameters()
        if parameter.requires_grad
    )
    vision_trainable = any(
        "vision_tower" in name and parameter.requires_grad
        for name, parameter in trainer.model.named_parameters()
    )
    assert not vision_trainable
    print(f"Trainable parameters: {trainable_parameters:,}")
    print("Vision tower trainable:", vision_trainable)

    train_output = trainer.train()
    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(ADAPTER_DIR))
    processor.tokenizer.save_pretrained(ADAPTER_DIR)

    elapsed_seconds = time.perf_counter() - training_started
    training_report = {
        **train_output.metrics,
        "profile": PROFILE,
        "model_id": MODEL_ID,
        "train_records": len(train_records),
        "validation_records": len(validation_records),
        "trainable_parameters": trainable_parameters,
        "elapsed_seconds": elapsed_seconds,
        "config": CONFIG,
        "hardware": hardware_snapshot(),
    }
    (OUTPUT_DIR / "training_metrics.json").write_text(
        json.dumps(training_report, indent=2),
        encoding="utf-8",
    )
    print(f"Adapter saved to {ADAPTER_DIR}")

    release_model_memory()
else:
    print("Set RUN_TRAINING = True and rerun this cell.")

Training on NVIDIA GeForce RTX 5060 Laptop GPU with 8.0 GiB VRAM.


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

c:\Users\money\anaconda3\envs\meddemo\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
C:\Users\money\AppData\Local\Temp\ipykernel_9040\1432907746.py:90: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(


Tokenizing train dataset (num_proc=1):   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=1):   0%|          | 0/100 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Trainable parameters: 14,901,248
Vision tower trainable: False


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,1.605168,1.656405,0.257407,0.651667,1208.000000
2,1.639829,1.184625,0.309789,0.689167,2384.000000
3,1.185928,0.225383,0.202557,0.933333,3537.000000
4,0.262977,0.149882,0.180623,0.945833,4693.000000
5,0.136064,0.160649,0.146530,0.945833,5897.000000
6,0.186098,0.108780,0.120848,0.957500,7103.000000
7,0.095274,0.089197,0.102084,0.968333,8309.000000
8,0.099963,0.079294,0.096976,0.974167,9515.000000
9,0.089170,0.084786,0.074078,0.966667,10727.000000
10,0.117552,0.067092,0.068867,0.967500,11934.000000


Adapter saved to c:\Users\money\OneDrive\Documents\GitHub\oss_model_clinical_triage_demo\outputs\notebook_larger_smoke_more_steps\adapter


## 8. Post-Fine-Tuning Evaluation

In [85]:
ADAPTER_EVALUATION_PATH = OUTPUT_DIR / "adapter_evaluation.json"

if RUN_ADAPTER_EVALUATION:
    if not ADAPTER_DIR.exists():
        raise FileNotFoundError(f"No adapter found at {ADAPTER_DIR}")

    release_model_memory()
    adapter_model, adapter_processor = load_inference_model(ADAPTER_DIR)
    adapter_predictions = evaluate_records(
        adapter_model,
        adapter_processor.tokenizer,
        test_records,
        "Fine-tuned MedGemma",
    )
    adapter_result = save_evaluation("adapter_evaluation", adapter_predictions)
    display_metrics(adapter_result)
    release_model_memory()
elif ADAPTER_EVALUATION_PATH.exists():
    adapter_result = json.loads(ADAPTER_EVALUATION_PATH.read_text(encoding="utf-8"))
    print("Loaded existing profile-specific adapter evaluation.")
    display_metrics(adapter_result)
else:
    print("Train an adapter, then set RUN_ADAPTER_EVALUATION = True.")


Before model load: 5.05 GiB allocated, 6.72 GiB reserved, 7.96 GiB total


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

c:\Users\money\anaconda3\envs\meddemo\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


KeyboardInterrupt: 

## 9. Base vs Fine-Tuned Clinical Metrics

In [83]:
def load_result(path: Path):
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None


base_result = load_result(BASE_EVALUATION_PATH)
adapter_result = load_result(ADAPTER_EVALUATION_PATH)

if base_result and adapter_result:
    if base_result["test_fingerprint"] != adapter_result["test_fingerprint"]:
        raise ValueError("Base and adapter evaluations used different test records.")

    metric_names = [
        "accuracy",
        "macro_f1",
        "coverage",
        "under_triage_rate",
        "over_triage_rate",
        "severe_under_triage_rate",
        "uncertainty_rate",
        "invalid_output_rate",
        "human_review_rate",
    ]
    comparison = pd.DataFrame(
        {
            "metric": metric_names,
            "base": [base_result["metrics"][name] for name in metric_names],
            "fine_tuned": [adapter_result["metrics"][name] for name in metric_names],
        }
    )
    comparison["change"] = comparison["fine_tuned"] - comparison["base"]
    display(comparison)

    recall_comparison = pd.DataFrame(
        {
            "base_recall": base_result["metrics"]["recall_by_esi"],
            "fine_tuned_recall": adapter_result["metrics"]["recall_by_esi"],
        }
    )
    display(recall_comparison)
else:
    print("Run both evaluations to create a fair comparison.")

,metric,base,fine_tuned,change
0,accuracy,0.200000,0.970000,0.770000
1,macro_f1,0.084211,0.969944,0.885733
2,coverage,1.000000,1.000000,0.000000
3,under_triage_rate,0.400000,0.000000,-0.400000
4,over_triage_rate,0.400000,0.030000,-0.370000
5,severe_under_triage_rate,0.250000,0.000000,-0.250000
6,uncertainty_rate,0.000000,0.000000,0.000000
7,invalid_output_rate,0.000000,0.000000,0.000000
8,human_review_rate,0.000000,0.000000,0.000000


,base_recall,fine_tuned_recall
1,0.0,1.00
2,0.0,1.00
3,1.0,0.95
4,0.0,1.00
5,0.0,0.90


## 9b. Error Analysis and Review Policy

This section turns the metric files into tables that are easy to inspect. It does not retrain or rerun inference.

The review policy is intentionally simple: invalid output, explicit `uncertain`, or low first-token confidence goes to human review. The confidence value is a rough model signal, not calibrated clinical probability.


In [84]:
def classify_error(row):
    actual = row.get("actual")
    prediction = row.get("prediction")
    if not isinstance(prediction, int):
        return "invalid_or_uncertain"
    if prediction == actual:
        return "correct"
    if actual in (1, 2) and prediction in (4, 5):
        return "severe_under_triage"
    if prediction > actual:
        return "under_triage"
    return "over_triage"


def make_error_table(result: dict) -> pd.DataFrame:
    rows = result.get("predictions", [])
    if not rows:
        return pd.DataFrame()

    table = pd.DataFrame(rows)
    table["correct"] = table["prediction"] == table["actual"]
    table["absolute_error"] = table.apply(
        lambda row: abs(row["prediction"] - row["actual"])
        if isinstance(row["prediction"], int)
        else None,
        axis=1,
    )
    table["error_type"] = table.apply(classify_error, axis=1)
    table["clinically_serious"] = table["error_type"].isin(
        ["severe_under_triage", "under_triage"]
    )

    preferred_columns = [
        "subject_id",
        "stay_id",
        "actual",
        "prediction",
        "correct",
        "error_type",
        "absolute_error",
        "clinically_serious",
        "prediction_confidence",
        "needs_human_review",
        "review_reason",
        "raw_response",
    ]
    columns = [column for column in preferred_columns if column in table.columns]
    return table[columns].sort_values(
        by=["correct", "clinically_serious", "absolute_error"],
        ascending=[True, False, False],
        na_position="last",
    )


def display_error_analysis(name: str, result: dict):
    table = make_error_table(result)
    if table.empty:
        print(f"No {name} predictions found.")
        return

    print(f"{name}: {int((~table['correct']).sum())} errors out of {len(table)} cases")
    display(table[~table["correct"]].head(30))

    if "needs_human_review" in table.columns:
        review_summary = (
            table.fillna({"review_reason": ""})
            .groupby(["needs_human_review", "review_reason"], dropna=False)
            .size()
            .reset_index(name="cases")
        )
        display(review_summary)


if base_result:
    display_error_analysis("Base model", base_result)

if adapter_result:
    display_error_analysis("Fine-tuned adapter", adapter_result)


Base model: 80 errors out of 100 cases


,subject_id,stay_id,actual,prediction,correct,error_type,absolute_error,clinically_serious,prediction_confidence,needs_human_review,review_reason,raw_response
6,10000396,30081418,1,5,False,severe_under_triage,4,True,0.923140,False,,"5\n```json\n{""predicted"
14,10000016,30009179,1,5,False,severe_under_triage,4,True,0.942642,False,,"5\n```json\n{""predicted"
22,10000023,30003782,1,5,False,severe_under_triage,4,True,0.926187,False,,5
31,10000035,30002714,1,5,False,severe_under_triage,4,True,0.933143,False,,"5\n```json\n{""predicted"
34,10000317,30070256,1,5,False,severe_under_triage,4,True,0.937349,False,,"5\n```json\n{""predicted"
43,10000182,30037124,1,5,False,severe_under_triage,4,True,0.950858,False,,"5\n```json\n{""predicted"
45,10000015,30017832,1,5,False,severe_under_triage,4,True,0.926549,False,,"5\n```json\n{""predicted"
46,10000396,30081758,1,5,False,severe_under_triage,4,True,0.913600,False,,"5\n```json\n{""predicted"
51,10000320,30065904,1,5,False,severe_under_triage,4,True,0.926797,False,,"5\n```json\n{""predicted"
53,10000317,30065423,1,5,False,severe_under_triage,4,True,0.928375,False,,"5\n```json\n{""predicted"


,needs_human_review,review_reason,cases
0,False,,100


Fine-tuned adapter: 3 errors out of 100 cases


,subject_id,stay_id,actual,prediction,correct,error_type,absolute_error,clinically_serious,prediction_confidence,needs_human_review,review_reason,raw_response
28,10000006,30005851,5,4,False,over_triage,1,False,0.999558,False,,4
37,10000006,30004203,5,4,False,over_triage,1,False,0.999617,False,,4
85,10000287,30053360,3,2,False,over_triage,1,False,0.999259,False,,2


,needs_human_review,review_reason,cases
0,False,,100


## 10. Hardware Report

In [86]:
environment_report = {
    "profile": PROFILE,
    "model_id": MODEL_ID,
    "config": CONFIG,
    "selected_data": {
        "train_rows": len(train_records),
        "validation_rows": len(validation_records),
        "test_rows": len(test_records),
        "test_fingerprint": test_fingerprint(test_records),
    },
    "hardware": hardware_snapshot(),
}

hardware_path = OUTPUT_DIR / "hardware_environment.json"
hardware_path.write_text(json.dumps(environment_report, indent=2), encoding="utf-8")

display(pd.json_normalize(environment_report))
print(f"Hardware report written to {hardware_path}")

,profile,model_id,config.data_dir,config.output_dir,config.train_limit,config.validation_limit,config.test_limit,config.max_steps,config.epochs,config.batch_size,...,hardware.packages.peft,hardware.packages.bitsandbytes,hardware.system_ram_gib,hardware.gpu.name,hardware.gpu.total_vram_gib,hardware.gpu.compute_capability,hardware.gpu.allocated_gib,hardware.gpu.reserved_gib,hardware.gpu.peak_allocated_gib,hardware.gpu.peak_reserved_gib
0,larger_smoke_more_steps,google/medgemma-1.5-4b-it,data/finetune_smoke,outputs/notebook_larger_smoke_more_steps,500,100,100,60,1,1,...,0.19.1,0.49.2,31.309132,NVIDIA GeForce RTX 5060 Laptop GPU,7.959534,"[12, 0]",7.074133,7.984375,8.202091,8.244141


Hardware report written to c:\Users\money\OneDrive\Documents\GitHub\oss_model_clinical_triage_demo\outputs\notebook_larger_smoke_more_steps\hardware_environment.json


## Interpretation

A successful smoke test confirms that model loading, QLoRA backpropagation, adapter saving, and adapter inference work on the selected hardware.

It does **not** establish clinical validity. The final report should emphasize:

- the source data is synthetic;
- `race` and `arrival_transport` are unavailable and represented as `UNKNOWN`;
- missing values are preserved;
- severe under-triage is more important than overall accuracy;
- real MIMIC-IV-ED evaluation requires authorized access and a separate validation study.